In [45]:
import requests #Pour faire des requêtes HTTP
from bs4 import BeautifulSoup #Pour extraire et manipuler des données HTML
import pandas as pd #Manipulation de données

In [62]:
categories_generiques = {"Accueil", "Femmes", "Hommes", "Enfants", "Vêtements"}

variables=["etat", "matiere", "couleur", "date", "prix", "prix_total", "categorie", "pays", "likes"]

df_vinted = pd.DataFrame(columns=variables)


In [47]:
url="https://www.vinted.fr/catalog?catalog[]=1904&brand_ids[]=12&page=1&time=1778184660"
response = requests.get(url,headers={"User-Agent": "Mozilla/5.0"})

# Vérification du statut de la requête
if response.status_code == 200: 
    print("Requête réussie; Code:",response.status_code)
else:
    print("Erreur :", response.status_code)

soup = BeautifulSoup(response.text, 'html.parser')


Requête réussie; Code: 200


In [48]:
urls = [link.get('href') for link in soup.find_all('a',class_=["new-item-box__overlay new-item-box__overlay--clickable" ]) if link.get('href')] 
print (urls[0])
print(len(urls))

https://www.vinted.fr/items/8851956521-jeans-zara?referrer=catalog
96


In [49]:
url=urls[0]
response = requests.get(url,headers={"User-Agent": "Mozilla/5.0"})

# Vérification du statut de la requête
if response.status_code == 200: 
    print("Requête réussie; Code:",response.status_code)
else:
    print("Erreur :", response.status_code)

soup = BeautifulSoup(response.text, 'html.parser')

Requête réussie; Code: 200


In [50]:
#scrapper l'état'
get_etat=soup.select_one('[itemprop="status"]')
etat = get_etat.get_text(" ", strip=True) if get_etat else None

etat

'Très bon état'

In [64]:
#scrapper la matière
get_matiere=soup.select_one('[itemprop="material"]')
if get_matiere:
    matiere = get_matiere.get_text(" ", strip=True)

matiere

In [52]:
#scrapper la couleur
get_couleur=soup.select_one('[itemprop="color"]')
couleur = get_couleur.get_text(" ", strip=True) if get_couleur else None

couleur

'Bleu clair'

In [53]:
#scrapper date
get_date=soup.select_one('[itemprop="upload_date"]')
date = get_date.get_text(" ", strip=True) if get_date else None

date

'Il y a 3 heures'

In [54]:
#scrapper le prix
get_prix = soup.select_one('[data-testid="item-price"]')
get_prix_total = soup.select_one('[data-testid="total-combined-price"]')

prix = get_prix.get_text(" ", strip=True) if get_prix else None
prix_total=get_prix_total.get_text(" ", strip=True) if get_prix else None

print(prix, prix_total)

8,00 € 9,10 €


In [55]:
#scrapper catégories

categories = [
    tag.get_text(" ", strip=True)
    for tag in soup.select('ul.breadcrumbs span[itemprop="title"]')
]

categorie = None

for cat in categories:
    if cat not in categories_generiques:
        categorie = cat
        break

print(categorie)

Jeans


In [56]:
#scrapper pays
get_pays = soup.select_one('[data-testid="seller-location"]')
pays=get_pays.get_text(" ", strip=True) if get_prix else None

pays


'Paris, France'

In [57]:
#scrapper likes
get_likes = soup.select_one('[data-testid="favourite-button"] span.web_ui__Text__text')
likes=get_likes.get_text(" ", strip=True) if get_likes else None

likes

'20'

In [72]:
def scrapping(link):
    response = requests.get(link,headers={"User-Agent": "Mozilla/5.0"})
    soup = BeautifulSoup(response.text, 'html.parser')
    infos=["status", "material", "color", "upload_date" ]
    #info
    values=[]
    for info in infos:
        get_info=soup.select_one(f'[itemprop="{info}"]')
        if get_info:
            info = get_info.get_text(" ", strip=True)
        else:
            info=None
        values.append(info)
    #prix
    get_prix = soup.select_one('[data-testid="item-price"]')
    get_prix_total = soup.select_one('[data-testid="total-combined-price"]')

    prix = get_prix.get_text(" ", strip=True) if get_prix else None
    prix_total=get_prix_total.get_text(" ", strip=True) if get_prix_total else None
    #categories
    categories = [
    tag.get_text(" ", strip=True)
    for tag in soup.select('ul.breadcrumbs span[itemprop="title"]')
    ]

    categorie = None

    for cat in categories:
        if cat not in categories_generiques:
            categorie = cat
            break
    #pays
    get_pays = soup.select_one('[data-testid="seller-location"]')
    if get_pays:
        pays=get_pays.get_text(" ", strip=True)
    else:
        pays=None

    #likes
    get_likes = soup.select_one('[data-testid="favourite-button"] span.web_ui__Text__text')
    if get_likes:
        likes=get_likes.get_text(" ", strip=True)
    else:
        likes=None
            
    values+=[prix, prix_total, categorie, pays, likes]

    return values
    

In [73]:
scrapping(url)

['Très bon état',
 None,
 'Bleu clair',
 'Il y a 3 heures',
 '8,00\xa0€',
 '9,10\xa0€',
 'Jeans',
 'Paris, France',
 '27']

In [74]:
#construction du dataframe
for i in urls:
    ligne=scrapping(i)
    df_vinted.loc[len(df_vinted)] = ligne
    
df_vinted.head()

,etat,matiere,couleur,date,prix,prix_total,categorie,pays,likes
0,Très bon état,None,Bleu clair,Il y a 3 heures,"8,00 €","8,91 €",Jeans,"Paris, France",21
1,Très bon état,None,Bleu clair,Il y a 3 heures,"8,00 €","9,10 €",Jeans,"Paris, France",26
2,Très bon état,None,Bleu clair,Il y a 3 heures,"8,00 €","9,10 €",Jeans,"Paris, France",27
3,Neuf sans étiquette,None,Bleu clair,Il y a une heure,"3,00 €","3,85 €",Jeans,None,16
4,Très bon état,None,Gris,Il y a 13 heures,"5,00 €","5,95 €",Jeans,"Vergèze, France",73


In [ ]:
df_vinted.to_csv("vinted_data.csv", index=False, encoding="utf-8-sig")

